In [6]:
import os
import numpy as np
import tensorflow as tf
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import classification_report
from hand_processing import process_images
from tensorflow.keras.callbacks import EarlyStopping

BASE_DIR = "ASL_Alphabet_Dataset/asl_alphabet_train/"
MAX_IMAGES = 2000
TEST_SIZE = 0.2
BATCH_SIZE = 32
EPOCHS = 100000


In [ ]:
X, y = process_images(BASE_DIR, MAX_IMAGES)
np.save('features.npy', X)
np.save('labels.npy', y)

In [7]:
X = np.load('features.npy')
y = np.load('labels.npy')
print(X.shape)
print(y.shape)

(36880, 69)
(36880,)


In [10]:
label_encoder = LabelEncoder()
encoded_labels = label_encoder.fit_transform(y)
joblib.dump(label_encoder, 'label_encoder.pkl')

['label_encoder.pkl']

In [17]:
X_train, X_test, y_train, y_test = train_test_split(X, encoded_labels, test_size=TEST_SIZE, random_state=42)


early_stopping = EarlyStopping(
    monitor='val_loss',  
    patience=20,          
    restore_best_weights=True  
)

model = tf.keras.Sequential([
    tf.keras.Input(shape=(X_train.shape[1],)),
    tf.keras.layers.Dense(128, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(64, activation='relu'),
    tf.keras.layers.Dropout(0.3),
    tf.keras.layers.Dense(len(np.unique(encoded_labels)), activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stopping] 
)

Epoch 1/100000
922/922 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.5580 - loss: 1.5173 - val_accuracy: 0.9276 - val_loss: 0.2903
Epoch 2/100000
922/922 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.8755 - loss: 0.4281 - val_accuracy: 0.9471 - val_loss: 0.2019
Epoch 3/100000
922/922 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9107 - loss: 0.3206 - val_accuracy: 0.9539 - val_loss: 0.1602
Epoch 4/100000
922/922 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9217 - loss: 0.2806 - val_accuracy: 0.9608 - val_loss: 0.1374
Epoch 5/100000
922/922 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9331 - loss: 0.2329 - val_accuracy: 0.9638 - val_loss: 0.1234
Epoch 6/100000
922/922 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9437 - loss: 0.2042 - val_accuracy: 0.9681 - val_loss: 0.1101
Epoch 7/100000
922/922 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9467 - loss: 0.1888 - val_accuracy: 0.9675 - val_loss: 0.1093
Epoch 8/100000
922/922 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9488 -

In [18]:
loss, accuracy = model.evaluate(X_test, y_test, verbose=2)
print(f"Test Loss: {loss}, Test Accuracy: {accuracy}")

y_pred = np.argmax(model.predict(X_test), axis=1)
print(classification_report(y_test, y_pred, target_names=label_encoder.classes_))

231/231 - 0s - 810us/step - accuracy: 0.9848 - loss: 0.0540
Test Loss: 0.05401068553328514, Test Accuracy: 0.9848155975341797
231/231 ━━━━━━━━━━━━━━━━━━━━ 0s 601us/step
              precision    recall  f1-score   support

           A       0.97      0.99      0.98       235
           B       1.00      1.00      1.00       297
           C       0.97      0.99      0.98       236
           D       1.00      0.96      0.98       318
           E       1.00      0.98      0.99       283
           F       1.00      0.99      1.00       387
           G       0.98      1.00      0.99       320
           H       0.99      0.99      0.99       319
           I       0.98      0.98      0.98       290
           J       0.99      0.98      0.98       305
           K       0.99      0.99      0.99       340
           L       1.00      1.00      1.00       342
           M       0.92      0.96      0.94       195
           N       0.98      0.92      0.95       143
           O       0

In [19]:
model.save("ASL_model.keras")